# CKD evidence landscape

This dashboard-ready notebook presents three separate, public-observed aggregates: MEPS HC-243 2022 person-year utilization/expenditure estimates, a bounded CMS Medicare Part D 2024 therapy dictionary, and a fully paginated ClinicalTrials.gov CKD registry snapshot. Each section retains its own population, time, grain, denominator, and limitations. No person, beneficiary, provider, or study identifier is joined across sources.

The fixture section at the end is explicitly `fixture_only` for offline contract demonstrations. It cannot overwrite or be substituted for the official aggregate artifacts.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
def load_verified(stem):
    artifact_path = ROOT / 'data' / 'processed' / f'{stem}.json'
    checksum_path = ROOT / 'data' / 'processed' / f'{stem}.sha256'
    manifest_path = ROOT / 'data' / 'manifests' / f'{stem}.json'
    digest = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
    assert digest in checksum_path.read_text(encoding='utf-8')
    artifact = json.loads(artifact_path.read_text(encoding='utf-8'))
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert artifact['retrieved_at'] == manifest['retrieved_at']
    return artifact, manifest

meps, meps_manifest = load_verified('meps_hc243_2022_landscape')
partd, partd_manifest = load_verified('partd_2024_ckd_therapy_landscape')
trials, trials_manifest = load_verified('clinicaltrials_ckd_landscape')
pd.DataFrame([{'source': x['source'], 'release': x['release'], 'evidence_type': x['evidence_type'], 'retrieved_at': x['retrieved_at']} for x in (meps, partd, trials)])

## MEPS: survey-weighted utilization and expenditures

These estimates use `DIABW22F` with `VARSTR`/`VARPSU` for the documented DCS diabetes-related kidney-problem proxy. `DSKIDN53` is not a confirmed CKD diagnosis.

In [ ]:
pd.DataFrame(meps['estimates'])[['metric', 'unit', 'point', 'standard_error', 'ci_low', 'ci_high', 'denominator', 'sum_weights']]

In [ ]:
pd.Series(meps['denominators'], name='MEPS exact record denominators').to_frame()

## CMS Part D: bounded therapy dictionary

The table is a selected, complete-for-the-dictionary provider-drug-state slice. CMS excludes providers with fewer than 11 total claims; absent rows are not zeros.

In [ ]:
pd.DataFrame(partd['dictionary'])[['generic_name', 'class']]
pd.DataFrame(partd['aggregates']).head(20)

In [ ]:
pd.DataFrame(partd['pagination']['generic_pages'])

## ClinicalTrials.gov: registry landscape

The exact CKD query is fully paginated. Status, phase/type, sponsor, geography, and update-year tables summarize registered studies only. Missing enrollment remains null in status/phase rows and is reported separately.

In [ ]:
print(trials['query'])
pd.Series(trials['counts'], name='ClinicalTrials.gov reconciliation').to_frame()

In [ ]:
status = pd.DataFrame(trials['status'])
phase_or_type = pd.DataFrame(trials['phase_or_type'])
sponsor = pd.DataFrame(trials['sponsor']).sort_values('study_count', ascending=False).head(15)
geography = pd.DataFrame(trials['geography']).sort_values('study_count', ascending=False).head(15)
change_over_time = pd.DataFrame(trials['change_over_time'])
display(status); display(phase_or_type); display(sponsor); display(geography); display(change_over_time)

## Decision boundaries

- MEPS describes a weighted survey population and its person-year utilization/expenditures; it is not provider prescribing data.
- Part D describes selected provider-drug aggregates; it is not an indication, adherence, outcomes, or beneficiary dataset.
- ClinicalTrials.gov describes registry submissions; recruitment status and phase are not proof of active enrollment, efficacy, or market share.
- These evidence cards may be displayed together for context, but no combined patient-level score or causal/commercial claim is valid without compatible populations, time windows, and analytic denominators.

## Offline fixture-only contract check

CI fixtures are intentionally separate and are not used to populate the official outputs above.

In [ ]:
from ckd_intelligence.ingestion.trials import ingest_trials
fixture = ingest_trials(ROOT / 'tests' / 'fixtures' / 'ingestion' / 'clinicaltrials.json', cache_dir=ROOT / 'data' / 'raw' / 'task5-fixture-cache')
fixture_rows = [{**dict(row), 'evidence_type': 'fixture_only'} for row in fixture.valid]
assert {row['evidence_type'] for row in fixture_rows} == {'fixture_only'}
print({'fixture_only_notebook_namespace': True, 'records': len(fixture_rows), 'official_artifacts_unchanged': True})